# Tinyllama

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,   # good for CPU stability
).to(device)

model.eval()

# Fix pad token
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

messages = [
    {"role": "system", "content": "You are a friendly and helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(device)

# Create clean generation config
gen_config = GenerationConfig.from_model_config(model.config)

# Important: remove inherited max_length warning
gen_config.max_length = None
gen_config.max_new_tokens = 512

# Deterministic generation
gen_config.do_sample = False
gen_config.repetition_penalty = 1.1
gen_config.pad_token_id = tokenizer.eos_token_id
gen_config.eos_token_id = tokenizer.eos_token_id

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        generation_config=gen_config
    )

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[-1]:],
    skip_special_tokens=True
)

print(response.strip())

The capital of France is Paris, located in the Île-de-France region.
